# Data Preparation

**Experiment:** Data Prep

Dataset splitting by patient ID to prevent data leakage between train/test sets.

---

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# split_by_patient.py
# -*- coding: utf-8 -*-
"""
Split por patient_id con empaquetado a .zip (train.zip y test.zip).

Modos de materialización:
  - symlink (por defecto en Linux/Mac)
  - hardlink (Windows si no hay permisos de symlink)
  - copy / move

Zipeo:
  - --zip_outputs: genera train.zip y test.zip bajo --out
  - --zip_mode staging: crea copias temporales para asegurar que el .zip contenga bytes reales (recomendado si usas symlinks)
  - --zip_mode direct: comprime las carpetas tal cual (puede guardar symlinks)
  - --remove_dirs_after_zip: elimina train/ y test/ al finalizar (conserva solo los .zip)

Ejemplo:
python split_by_patient.py --root "/data/datasetclean" --out "/data/splits/by_patient_30test" \
  --test_ratio 0.30 --seed 42 --mode symlink --pid_regex "^(.*?)_" \
  --zip_outputs --zip_mode staging --remove_dirs_after_zip
"""
import os, re, csv, shutil, argparse, sys
from pathlib import Path
from typing import Tuple, List, Optional, Iterable
from dataclasses import dataclass
import zipfile
import random
import numpy as np

try:
    from sklearn.model_selection import GroupShuffleSplit
except Exception as e:
    print("[ERROR] scikit-learn es requerido para GroupShuffleSplit:", e)
    sys.exit(1)

IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

@dataclass
class Item:
    path: Path
    cls: str
    pid: str

def extract_pid(name: str, pid_regex: Optional[str]) -> str:
    if pid_regex:
        m = re.match(pid_regex, name)
        if m: return m.group(1)
    # fallback: antes del primer "_"
    return name.split("_")[0]

def scan_dataset(root: Path, pid_regex: Optional[str]) -> Tuple[List[Item], List[str]]:
    assert root.is_dir(), f"No es carpeta: {root}"
    classes = sorted([d.name for d in root.iterdir() if d.is_dir()])
    items: List[Item] = []
    for cname in classes:
        cdir = root / cname
        for p in cdir.rglob("*"):
            if p.suffix.lower() in IMG_EXTS and p.is_file():
                pid = extract_pid(p.stem, pid_regex)
                items.append(Item(path=p, cls=cname, pid=pid))
    return items, classes

def group_split(items: List[Item], test_ratio: float, seed: int) -> Tuple[List[Item], List[Item]]:
    X = np.arange(len(items))
    groups = np.array([it.pid for it in items])
    gss = GroupShuffleSplit(n_splits=1, test_size=test_ratio, random_state=seed)
    tr_idx, te_idx = next(gss.split(X, groups=groups))
    train = [items[i] for i in tr_idx]
    test  = [items[i] for i in te_idx]
    # sanity: sin traslape de pacientes
    assert set(it.pid for it in train).isdisjoint(set(it.pid for it in test)), "Overlap de PIDs!"
    return train, test

def _ensure_dir(d: Path): d.mkdir(parents=True, exist_ok=True)

def _materialize(src: Path, dst: Path, mode: str):
    _ensure_dir(dst.parent)
    if mode == "symlink":
        try:
            if dst.exists(): dst.unlink()
            dst.symlink_to(src)
        except Exception:
            # fallback a hardlink si symlink falla (Windows sin permisos)
            if dst.exists(): dst.unlink()
            os.link(src, dst)
    elif mode == "hardlink":
        if dst.exists(): dst.unlink()
        os.link(src, dst)
    elif mode == "copy":
        shutil.copy2(src, dst)
    elif mode == "move":
        shutil.move(str(src), str(dst))
    else:
        raise ValueError(f"Modo desconocido: {mode}")

def write_split(
    subset: List[Item], out_root: Path, mode: str, dry_run: bool = False
) -> List[Tuple[Path, Path]]:
    pairs = []
    for it in subset:
        rel = Path(it.cls) / it.path.name
        dst = out_root / rel
        pairs.append((it.path, dst))
        if not dry_run:
            _ensure_dir(dst.parent)
            _materialize(it.path, dst, mode)
    return pairs

def save_manifest(
    manifest_path: Path,
    rows: Iterable[Tuple[str, str, str, str, str]]
):
    _ensure_dir(manifest_path.parent)
    with open(manifest_path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["split","class","patient_id","src","dst","action"])
        for row in rows: w.writerow(row)

def copy_tree(src_dir: Path, dst_dir: Path):
    for root, _, files in os.walk(src_dir):
        rel = os.path.relpath(root, src_dir)
        for fn in files:
            src = Path(root) / fn
            dst = dst_dir / rel / fn
            _ensure_dir(dst.parent)
            shutil.copy2(src, dst)

def make_zip_from_dir(src_dir: Path, zip_path: Path):
    _ensure_dir(zip_path.parent)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk(src_dir):
            rel_root = os.path.relpath(root, src_dir)
            for fn in files:
                p = Path(root) / fn
                arc = (Path(rel_root) / fn) if rel_root != "." else Path(fn)
                z.write(p, arcname=str(arc))

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)

def split_by_patient(
    root: Path, out_dir: Path, test_ratio: float = 0.30, seed: int = 42,
    mode: str = "symlink", pid_regex: Optional[str] = r"^(.*?)_",
    dry_run: bool = False,
    zip_outputs: bool = False, zip_mode: str = "staging", remove_dirs_after_zip: bool = False
):
    set_seed(seed)
    items, classes = scan_dataset(root, pid_regex)
    print(f"[SCAN] {len(items)} imágenes | clases={classes}")

    train, test = group_split(items, test_ratio=test_ratio, seed=seed)
    print(f"[SPLIT] train={len(train)} | test={len(test)} | test_ratio={test_ratio:.2f}")

    train_dir = out_dir / "train"
    test_dir  = out_dir / "test"
    if not dry_run:
        if remove_dirs_after_zip and out_dir.exists():
            # limpieza explícita si se pidió reescritura total
            pass
        _ensure_dir(train_dir); _ensure_dir(test_dir)

    # Materializar
    train_pairs = write_split(train, train_dir, mode=mode, dry_run=dry_run)
    test_pairs  = write_split(test,  test_dir,  mode=mode, dry_run=dry_run)

    # Manifest
    rows = []
    for src, dst in train_pairs:
        pid = extract_pid(src.stem, pid_regex)
        rows.append(("train", dst.parent.name, pid, str(src), str(dst), mode))
    for src, dst in test_pairs:
        pid = extract_pid(src.stem, pid_regex)
        rows.append(("test", dst.parent.name, pid, str(src), str(dst), mode))
    save_manifest(out_dir / "manifest_split_by_patient.csv", rows)

    # Zipeo opcional
    if zip_outputs and not dry_run:
        print(f"[ZIP] Modo de zipeo: {zip_mode}")
        if zip_mode not in {"staging", "direct"}:
            raise ValueError("--zip_mode debe ser 'staging' o 'direct'")

        if zip_mode == "staging":
            staging = out_dir / "_staging_zip"
            staging_train = staging / "train"
            staging_test  = staging / "test"
            if staging.exists():
                shutil.rmtree(staging)
            _ensure_dir(staging_train); _ensure_dir(staging_test)
            print("[ZIP] Copiando train/ a staging...")
            copy_tree(train_dir, staging_train)
            print("[ZIP] Copiando test/ a staging...")
            copy_tree(test_dir, staging_test)

            train_zip = out_dir / "train.zip"
            test_zip  = out_dir / "test.zip"
            print("[ZIP] Creando train.zip ...")
            make_zip_from_dir(staging_train, train_zip)
            print("[ZIP] Creando test.zip ...")
            make_zip_from_dir(staging_test, test_zip)

            print("[ZIP] Eliminando staging...")
            shutil.rmtree(staging, ignore_errors=True)

        else:  # direct
            print("[ZIP] Empaquetado directo (puede incluir symlinks).")
            train_zip = out_dir / "train.zip"
            test_zip  = out_dir / "test.zip"
            make_zip_from_dir(train_dir, train_zip)
            make_zip_from_dir(test_dir,  test_zip)

        print(f"[OK] ZIPs: {train_zip} | {test_zip}")

        if remove_dirs_after_zip:
            print("[CLEAN] Eliminando carpetas train/ y test/ tras zipeo...")
            shutil.rmtree(train_dir, ignore_errors=True)
            shutil.rmtree(test_dir,  ignore_errors=True)

    print(f"[DONE] Split escrito en: {out_dir}")
    return {"train_dir": train_dir, "test_dir": test_dir, "classes": classes}

In [ ]:
res = split_by_patient(
    root=Path("/content/drive/MyDrive/TESIS/datasetclean"),
    out_dir=Path("/content/drive/MyDrive/TESIS/splits/by_patient_30test1"),
    test_ratio=0.30,
    seed=42,
    mode="copy",                 # "symlink" | "hardlink" | "copy" | "move"
    pid_regex=r"^(.*?)_",           # ajusta si tu patrón de patient_id es otro
    dry_run=False,                  # True = no escribe, solo muestra
    zip_outputs=True,               # genera train.zip y test.zip
    zip_mode="staging",             # "staging" (recomendado) o "direct"
    remove_dirs_after_zip=True      # elimina carpetas train/ y test/ tras crear ZIPs
)

In [ ]:
# split_random.py
# -*- coding: utf-8 -*-
"""
Random split of images into train/test, ignoring patient_id.

Dataset structure (input):
    root/
        class_1/
            img1.png
            img2.png
            ...
        class_2/
            ...
Output:
    out/
        train/
            class_1/...
            class_2/...
        test/
            class_1/...
            class_2/...
        manifest_split_random.csv
        (optional) train.zip, test.zip

Example:
    python split_random.py --root "/data/datasetclean" --out "/data/splits/random_30test" \
        --test_ratio 0.30 --seed 42 --mode symlink \
        --zip_outputs --zip_mode staging --remove_dirs_after_zip
"""
import os
import csv
import shutil
import argparse
import sys
from pathlib import Path
from typing import Tuple, List, Iterable
from dataclasses import dataclass
import zipfile
import random
import numpy as np

IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}


@dataclass
class Item:
    path: Path
    cls: str


def scan_dataset(root: Path) -> Tuple[List[Item], List[str]]:
    """
    Scan dataset under 'root', assuming structure root/class_name/*.ext.
    Returns:
        items: list of Item(path, cls)
        classes: sorted list of class names
    """
    assert root.is_dir(), f"Not a directory: {root}"
    classes = sorted([d.name for d in root.iterdir() if d.is_dir()])
    items: List[Item] = []

    for cname in classes:
        cdir = root / cname
        for p in cdir.rglob("*"):
            if p.suffix.lower() in IMG_EXTS and p.is_file():
                items.append(Item(path=p, cls=cname))

    return items, classes


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)


def random_split(
    items: List[Item],
    test_ratio: float,
    seed: int
) -> Tuple[List[Item], List[Item]]:
    """
    Random split (ignoring patient_id). Each image is treated independently.

    Returns:
        train_items, test_items
    """
    set_seed(seed)
    n = len(items)
    indices = list(range(n))
    random.shuffle(indices)

    n_test = int(round(test_ratio * n))
    test_idx = set(indices[:n_test])
    train_idx = set(indices[n_test:])

    train = [items[i] for i in range(n) if i in train_idx]
    test = [items[i] for i in range(n) if i in test_idx]

    return train, test


def _ensure_dir(d: Path):
    d.mkdir(parents=True, exist_ok=True)


def _materialize(src: Path, dst: Path, mode: str):
    _ensure_dir(dst.parent)
    if mode == "symlink":
        try:
            if dst.exists():
                dst.unlink()
            dst.symlink_to(src)
        except Exception:
            # fallback to hardlink if symlink fails (e.g. Windows)
            if dst.exists():
                dst.unlink()
            os.link(src, dst)
    elif mode == "hardlink":
        if dst.exists():
            dst.unlink()
        os.link(src, dst)
    elif mode == "copy":
        shutil.copy2(src, dst)
    elif mode == "move":
        shutil.move(str(src), str(dst))
    else:
        raise ValueError(f"Unknown mode: {mode}")


def write_split(
    subset: List[Item],
    out_root: Path,
    mode: str,
    dry_run: bool = False
):
    pairs = []
    for it in subset:
        rel = Path(it.cls) / it.path.name
        dst = out_root / rel
        pairs.append((it.path, dst))
        if not dry_run:
            _ensure_dir(dst.parent)
            _materialize(it.path, dst, mode)
    return pairs


def save_manifest(
    manifest_path: Path,
    rows: Iterable[tuple]
):
    """
    Manifest columns:
      split, class, src, dst, action
    (No patient_id here, because we ignore it.)
    """
    _ensure_dir(manifest_path.parent)
    with open(manifest_path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["split", "class", "src", "dst", "action"])
        for row in rows:
            w.writerow(row)


def copy_tree(src_dir: Path, dst_dir: Path):
    for root, _, files in os.walk(src_dir):
        rel = os.path.relpath(root, src_dir)
        for fn in files:
            src = Path(root) / fn
            dst = dst_dir / rel / fn
            _ensure_dir(dst.parent)
            shutil.copy2(src, dst)


def make_zip_from_dir(src_dir: Path, zip_path: Path):
    _ensure_dir(zip_path.parent)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk(src_dir):
            rel_root = os.path.relpath(root, src_dir)
            for fn in files:
                p = Path(root) / fn
                if rel_root == ".":
                    arc = Path(fn)
                else:
                    arc = Path(rel_root) / fn
                z.write(p, arcname=str(arc))


def split_random(
    root: Path,
    out_dir: Path,
    test_ratio: float = 0.30,
    seed: int = 42,
    mode: str = "symlink",
    dry_run: bool = False,
    zip_outputs: bool = False,
    zip_mode: str = "staging",
    remove_dirs_after_zip: bool = False
):
    """
    High-level function to perform random split and optional zipping.
    """
    items, classes = scan_dataset(root)
    print(f"[SCAN] {len(items)} images | classes={classes}")

    train, test = random_split(items, test_ratio=test_ratio, seed=seed)
    print(f"[SPLIT] train={len(train)} | test={len(test)} | test_ratio={test_ratio:.2f}")

    train_dir = out_dir / "train"
    test_dir = out_dir / "test"

    if not dry_run:
        _ensure_dir(train_dir)
        _ensure_dir(test_dir)

    # Materialize
    train_pairs = write_split(train, train_dir, mode=mode, dry_run=dry_run)
    test_pairs = write_split(test, test_dir, mode=mode, dry_run=dry_run)

    # Manifest
    rows = []
    for src, dst in train_pairs:
        rows.append(("train", dst.parent.name, str(src), str(dst), mode))
    for src, dst in test_pairs:
        rows.append(("test", dst.parent.name, str(src), str(dst), mode))

    save_manifest(out_dir / "manifest_split_random.csv", rows)

    # Optional zipping
    if zip_outputs and not dry_run:
        print(f"[ZIP] Zip mode: {zip_mode}")
        if zip_mode not in {"staging", "direct"}:
            raise ValueError("--zip_mode must be 'staging' or 'direct'")

        if zip_mode == "staging":
            staging = out_dir / "_staging_zip"
            staging_train = staging / "train"
            staging_test = staging / "test"
            if staging.exists():
                shutil.rmtree(staging)
            _ensure_dir(staging_train)
            _ensure_dir(staging_test)

            print("[ZIP] Copying train/ to staging...")
            copy_tree(train_dir, staging_train)
            print("[ZIP] Copying test/ to staging...")
            copy_tree(test_dir, staging_test)

            train_zip = out_dir / "train.zip"
            test_zip = out_dir / "test.zip"
            print("[ZIP] Creating train.zip ...")
            make_zip_from_dir(staging_train, train_zip)
            print("[ZIP] Creating test.zip ...")
            make_zip_from_dir(staging_test, test_zip)

            print("[ZIP] Removing staging...")
            shutil.rmtree(staging, ignore_errors=True)

        else:  # direct
            print("[ZIP] Direct zipping (may include symlinks).")
            train_zip = out_dir / "train.zip"
            test_zip = out_dir / "test.zip"
            make_zip_from_dir(train_dir, train_zip)
            make_zip_from_dir(test_dir, test_zip)

        print(f"[OK] ZIPs created: {train_zip} | {test_zip}")

        if remove_dirs_after_zip:
            print("[CLEAN] Removing train/ and test/ after zipping...")
            shutil.rmtree(train_dir, ignore_errors=True)
            shutil.rmtree(test_dir, ignore_errors=True)

    print(f"[DONE] Random split written to: {out_dir}")
    return {"train_dir": train_dir, "test_dir": test_dir, "classes": classes}


def parse_args():
    p = argparse.ArgumentParser(
        description="Random train/test split by image (ignoring patient_id)."
    )
    p.add_argument("--root", type=str, required=True,
                   help="Root dataset directory (with class subfolders).")
    p.add_argument("--out", type=str, required=True,
                   help="Output directory for split (train/, test/).")
    p.add_argument("--test_ratio", type=float, default=0.30,
                   help="Test ratio (default 0.30).")
    p.add_argument("--seed", type=int, default=42,
                   help="Random seed.")
    p.add_argument("--mode", type=str, default="symlink",
                   choices=["symlink", "hardlink", "copy", "move"],
                   help="Materialization mode for files.")
    p.add_argument("--dry_run", action="store_true",
                   help="Do not write files, only print what would happen.")
    p.add_argument("--zip_outputs", action="store_true",
                   help="If set, creates train.zip and test.zip under --out.")
    p.add_argument("--zip_mode", type=str, default="staging",
                   choices=["staging", "direct"],
                   help="Zip mode: 'staging' (copy to tmp before) or 'direct'.")
    p.add_argument("--remove_dirs_after_zip", action="store_true",
                   help="Remove train/ and test/ after creating ZIPs.")
    return p.parse_args()

In [ ]:
# (2) Rutas
ZIP ="/content/drive/MyDrive/TESIS/datasetclean.zip"
DEST = "/content/localdata"                 # carpeta base local
UNZIP_DIR = f"{DEST}/trainori_unzipped"        # donde se extraerá el zip

import os, shutil, pathlib, subprocess

# (3) Crear carpeta destino y copiar el ZIP localmente (I/O mucho más rápido)
os.makedirs(DEST, exist_ok=True)
local_zip = f"{DEST}/trainori.zip"
shutil.copy2(ZIP, local_zip)
print("ZIP copiado a:", local_zip)

# (4) Descomprimir (silencioso, sobrescribe si ya existe)
os.makedirs(UNZIP_DIR, exist_ok=True)
!unzip -q -o "/content/localdata/trainori.zip" -d "/content/localdata/trainori_unzipped"

# (5) (Opcional) eliminar carpeta __MACOSX si existe
!find "/content/localdata/trainori_unzipped" -type d -name "__MACOSX" -prune -exec rm -rf {} +

# (6) Mostrar estructura principal para que ubiques DATA_ROOT / TEST_ROOT
print("Contenido de", UNZIP_DIR)
!ls -lah "/content/localdata/trainori_unzipped" | head -n 50

In [ ]:
import os
from pathlib import Path

def check_image_sizes(root_dir: Path):
    IMG_EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
    zero_byte_images = []
    print(f"[INFO] Checking image sizes in: {root_dir}")

    if not root_dir.is_dir():
        print(f"[ERROR] Directory not found: {root_dir}")
        return

    for dirpath, _, filenames in os.walk(root_dir):
        for filename in filenames:
            file_path = Path(dirpath) / filename
            if file_path.suffix.lower() in IMG_EXTS:
                if os.path.getsize(file_path) == 0:
                    zero_byte_images.append(str(file_path))

    if zero_byte_images:
        print("[WARNING] Found 0-byte image files:")
        for img_path in zero_byte_images:
            print(f"- {img_path}")
    else:
        print("[SUCCESS] All image files found have a size greater than 0 bytes.")

# Directorio a revisar
directory_to_check = Path("/content/localdata/trainori_unzipped")
check_image_sizes(directory_to_check)

In [ ]:
from pathlib import Path

root = Path("/content/localdata/trainori_unzipped/datasetclean")
out_dir = Path("/content/drive/MyDrive/TESIS/StyleGan2/splits/random_30testleak")

split_info = split_random(
    root=root,
    out_dir=out_dir,
    test_ratio=0.30,
    seed=122,
    mode="copy",         # o "copy" si estás en Windows y no quieres líos
    zip_outputs=True,
    zip_mode="staging",
    remove_dirs_after_zip=False,
)

print(split_info)

In [ ]:
import os
import csv
import shutil
import argparse
import zipfile
from pathlib import Path
from typing import List, Tuple

import numpy as np
from sklearn.model_selection import train_test_split

IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}


def unzip_to_temp(zip_path: Path, out_dir: Path) -> Path:
    """
    Extracts the zip file into out_dir/_unzipped and returns that path.
    If _unzipped exists, it is removed first.
    """
    temp_root = out_dir / "_unzipped"
    if temp_root.exists():
        shutil.rmtree(temp_root, ignore_errors=True)
    temp_root.mkdir(parents=True, exist_ok=True)

    print(f"[UNZIP] Extracting {zip_path} to {temp_root}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(temp_root)

    return temp_root


def detect_dataset_root(unzipped_root: Path) -> Path:
    """
    Heuristic to find the dataset root:
    - If unzipped_root directly contains class folders with images, use it.
    - Otherwise, if there is exactly one child directory, descend into it.
    """
    subdirs = [d for d in unzipped_root.iterdir() if d.is_dir()]
    if len(subdirs) == 0:
        raise RuntimeError(f"No subdirectories found in {unzipped_root}")

    # Check if unzipped_root has subfolders with images
    if _looks_like_dataset_root(unzipped_root):
        print(f"[ROOT] Using {unzipped_root} as dataset root.")
        return unzipped_root

    # If not, but there is only one child dir, try that
    if len(subdirs) == 1:
        candidate = subdirs[0]
        if _looks_like_dataset_root(candidate):
            print(f"[ROOT] Using {candidate} as dataset root.")
            return candidate

    # Fallback: raise error
    raise RuntimeError(
        f"Could not detect dataset root under {unzipped_root}. "
        f"Expected a folder with class subfolders containing images."
    )


def _looks_like_dataset_root(root: Path) -> bool:
    """
    Returns True if 'root' contains >= 2 subdirectories, each with at least one image file.
    """
    class_dirs = [d for d in root.iterdir() if d.is_dir()]
    if len(class_dirs) < 2:
        return False

    has_images = 0
    for cdir in class_dirs:
        for p in cdir.rglob("*"):
            if p.is_file() and p.suffix.lower() in IMG_EXTS:
                has_images += 1
                break
    return has_images >= 2


def scan_images(dataset_root: Path) -> Tuple[List[Path], List[str], List[str]]:
    """
    Scans dataset_root/<class>/* and returns:
      - paths: list of image paths
      - labels: list of class names (one per image)
      - class_names: sorted list of class names
    """
    class_names = sorted([d.name for d in dataset_root.iterdir() if d.is_dir()])
    if len(class_names) == 0:
        raise RuntimeError(f"No class subfolders found in {dataset_root}")

    print(f"[SCAN] Classes found: {class_names}")

    paths: List[Path] = []
    labels: List[str] = []

    for cname in class_names:
        cdir = dataset_root / cname
        n_c = 0
        for p in cdir.rglob("*"):
            if p.is_file() and p.suffix.lower() in IMG_EXTS:
                paths.append(p)
                labels.append(cname)
                n_c += 1
        print(f"[SCAN] Class '{cname}': {n_c} images")

    print(f"[SCAN] Total images: {len(paths)}")
    return paths, labels, class_names


def materialize_split(
    split_paths: List[Path],
    split_labels: List[str],
    out_root: Path,
    split_name: str
) -> List[Tuple[str, str, str]]:
    """
    Copies images into out_root/<split_name>/<class>/filename.

    Returns a list of rows: (split, class, src, dst).
    """
    rows: List[Tuple[str, str, str]] = []
    for src, cname in zip(split_paths, split_labels):
        dst_dir = out_root / split_name / cname
        dst_dir.mkdir(parents=True, exist_ok=True)
        dst = dst_dir / src.name
        shutil.copy2(src, dst)
        rows.append((split_name, cname, str(src), str(dst)))
    return rows


def save_manifest(manifest_path: Path, rows: List[Tuple[str, str, str]]):
    """
    Saves a CSV manifest with columns: split,class,src,dst
    """
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    with open(manifest_path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["split", "class", "src", "dst"])
        for row in rows:
            w.writerow(row)
    print(f"[MANIFEST] Saved to {manifest_path}")


def make_zip_from_dir(src_dir: Path, zip_path: Path):
    """
    Creates a .zip from the contents of src_dir.
    """
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    print(f"[ZIP] Creating {zip_path} from {src_dir}")
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk(src_dir):
            rel_root = os.path.relpath(root, src_dir)
            for fn in files:
                p = Path(root) / fn
                if rel_root == ".":
                    arcname = fn
                else:
                    arcname = str(Path(rel_root) / fn)
                z.write(p, arcname=arcname)


def random_split_from_zip(
    zip_path: Path,
    out_dir: Path,
    test_ratio: float = 0.30,
    seed: int = 42,
):
    """
    Main function: unzip, detect root, random stratified split, copy files,
    then create train.zip and test.zip.
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    # 1) Unzip
    unzipped_root = unzip_to_temp(zip_path, out_dir)

    # 2) Detect dataset root
    dataset_root = detect_dataset_root(unzipped_root)

    # 3) Scan images
    paths, labels, class_names = scan_images(dataset_root)

    # 4) Random stratified split (by image, patient leakage allowed)
    paths_train, paths_test, labels_train, labels_test = train_test_split(
        paths,
        labels,
        test_size=test_ratio,
        random_state=seed,
        stratify=labels,
    )

    print(f"[SPLIT] train={len(paths_train)} | test={len(paths_test)} | test_ratio={test_ratio:.2f}")

    train_dir = out_dir / "train"
    test_dir = out_dir / "test"

    # <--- FIX START --->
    # Limpiar directorios de salida antes de materializar el split para evitar acumulación de archivos
    if train_dir.exists():
        print(f"[CLEAN] Eliminando directorio existente: {train_dir}")
        shutil.rmtree(train_dir)
    if test_dir.exists():
        print(f"[CLEAN] Eliminando directorio existente: {test_dir}")
        shutil.rmtree(test_dir)
    # <--- FIX END --->

    # 5) Materialize split (copy files)
    rows: List[Tuple[str, str, str]] = []
    rows += materialize_split(paths_train, labels_train, out_dir, "train")
    rows += materialize_split(paths_test, labels_test, out_dir, "test")

    # 6) Save manifest
    save_manifest(out_dir / "manifest_random_split.csv", rows)

    # 7) Create train.zip and test.zip from the generated folders
    train_zip = out_dir / "train.zip"
    test_zip = out_dir / "test.zip"

    if train_dir.exists():
        make_zip_from_dir(train_dir, train_zip)
    else:
        print("[ZIP] WARNING: train directory not found, skipping train.zip")

    if test_dir.exists():
        make_zip_from_dir(test_dir, test_zip)
    else:
        print("[ZIP] WARNING: test directory not found, skipping test.zip")

    print(f"[DONE] Random split written under: {out_dir}")
    print(f"[DONE] Zips: {train_zip} | {test_zip}")
    return {
        "train_dir": train_dir,
        "test_dir": test_dir,
        "classes": class_names,
        "train_zip": train_zip,
        "test_zip": test_zip,
    }


def build_argparser():
    p = argparse.ArgumentParser(description="Random train/test split from tumor ZIP (with patient leakage).")
    p.add_argument("--zip_path", type=str, required=True, help="Path to the .zip file (tumor dataset).")
    p.add_argument("--out_dir", type=str, required=True, help="Output directory for train/test split.")
    p.add_argument("--test_ratio", type=float, default=0.30, help="Test ratio (default: 0.30).")
    p.add_argument("--seed", type=int, default=42, help="Random seed (default: 42).")
    return p


def main():
    parser = build_argparser()
    args = parser.parse_args()

    zip_path = Path(args.zip_path)
    out_dir = Path(args.out_dir)

    random_split_from_zip(
        zip_path=zip_path,
        out_dir=out_dir,
        test_ratio=args.test_ratio,
        seed=args.seed,
    )

In [ ]:
import os
import shutil
from pathlib import Path
import zipfile

# --- 1. Definir rutas ---

# Rutas a los ZIPs generados (del último split aleatorio con patient leakage)
train_zip_gdrive = result['train_zip']
test_zip_gdrive = result['test_zip']

# Carpeta local de destino para los ZIPs y su extracción
local_dest_dir = Path("/content/local_extracted_splits")
local_dest_dir.mkdir(parents=True, exist_ok=True)

local_train_zip = local_dest_dir / "train.zip"
local_test_zip = local_dest_dir / "test.zip"

# Carpetas donde se extraerán los contenidos
unzipped_train_dir = local_dest_dir / "train_unzipped"
unzipped_test_dir = local_dest_dir / "test_unzipped"

IMG_EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}

def count_images_in_dir(directory: Path) -> int:
    """Cuenta el número de archivos de imagen en un directorio dado."""
    count = 0
    if not directory.is_dir():
        print(f"[ERROR] Directorio no encontrado: {directory}")
        return 0
    for root, _, files in os.walk(directory):
        for file in files:
            if Path(file).suffix.lower() in IMG_EXTS:
                count += 1
    return count

print(f"[INFO] Copiando {train_zip_gdrive} a {local_train_zip}...")
shutil.copy2(train_zip_gdrive, local_train_zip)
print(f"[INFO] Copiando {test_zip_gdrive} a {local_test_zip}...")
shutil.copy2(test_zip_gdrive, local_test_zip)

print(f"[INFO] Extrayendo {local_train_zip} a {unzipped_train_dir}...")
unzipped_train_dir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(local_train_zip, 'r') as zip_ref:
    zip_ref.extractall(unzipped_train_dir)

print(f"[INFO] Extrayendo {local_test_zip} a {unzipped_test_dir}...")
unzipped_test_dir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(local_test_zip, 'r') as zip_ref:
    zip_ref.extractall(unzipped_test_dir)

# --- 4. Contar imágenes ---

num_train_images = count_images_in_dir(unzipped_train_dir)
num_test_images = count_images_in_dir(unzipped_test_dir)

print(f"\n[RESULTADO] Imágenes en train_unzipped: {num_train_images}")
print(f"[RESULTADO] Imágenes en test_unzipped: {num_test_images}")

### **Paso 1: Definición de la Función `random_split_and_zip`**

Esta celda contiene la función principal que se encarga de todo el proceso:

1.  **Descomprimir el ZIP de entrada**: Extrae tu conjunto de datos original a una carpeta temporal.
2.  **Detectar la raíz del dataset**: Identifica automáticamente dónde están las carpetas de clases dentro del ZIP.
3.  **Escanear imágenes**: Recopila las rutas de todas las imágenes y sus etiquetas de clase.
4.  **Split Estratificado Aleatorio**: Divide las imágenes en conjuntos de entrenamiento y prueba (70/30 en este caso) de forma aleatoria, pero asegurándose de que la proporción de clases se mantenga similar en ambos conjuntos (`stratify=labels`). **Esta es la parte clave para el split aleatorio sin considerar ID de paciente.**
5.  **Materializar el Split**: Copia físicamente las imágenes a las carpetas `train/` y `test/` dentro de tu directorio de salida.
6.  **Crear Manifiesto**: Genera un archivo CSV (`manifest_random_split.csv`) que detalla qué imagen fue a qué split.
7.  **Zipear los Splits**: Comprime las carpetas `train/` y `test/` en `train.zip` y `test.zip`.
8.  **Limpiar**: Elimina las carpetas temporales y las carpetas `train/` y `test/` si se especifica.

In [ ]:
import os
import csv
import shutil
import zipfile
from pathlib import Path
from typing import List, Tuple, Dict

import numpy as np
from sklearn.model_selection import train_test_split

IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

def _unzip_to_temp(zip_path: Path, out_dir: Path) -> Path:
    temp_root = out_dir / "_unzipped_temp"
    if temp_root.exists():
        shutil.rmtree(temp_root)
    temp_root.mkdir(parents=True, exist_ok=True)
    print(f"[UNZIP] Extracting {zip_path} to {temp_root}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(temp_root)
    return temp_root

def _detect_dataset_root(unzipped_root: Path) -> Path:
    subdirs = [d for d in unzipped_root.iterdir() if d.is_dir()]
    if len(subdirs) == 0:
        raise RuntimeError(f"No subdirectories found in {unzipped_root}")

    def _looks_like_dataset_root(root: Path) -> bool:
        class_dirs = [d for d in root.iterdir() if d.is_dir()]
        if len(class_dirs) < 2:
            return False
        has_images_in_classes = 0
        for cdir in class_dirs:
            for p in cdir.rglob("*"):
                if p.is_file() and p.suffix.lower() in IMG_EXTS:
                    has_images_in_classes += 1
                    break
        return has_images_in_classes >= 2

    if _looks_like_dataset_root(unzipped_root):
        print(f"[ROOT] Using {unzipped_root} as dataset root.")
        return unzipped_root

    if len(subdirs) == 1:
        candidate = subdirs[0]
        if _looks_like_dataset_root(candidate):
            print(f"[ROOT] Using {candidate} as dataset root.")
            return candidate

    raise RuntimeError(
        f"Could not detect dataset root under {unzipped_root}. "
        f"Expected a folder with class subfolders containing images."
    )

def _scan_images(dataset_root: Path) -> Tuple[List[Path], List[str], List[str]]:
    class_names = sorted([d.name for d in dataset_root.iterdir() if d.is_dir()])
    if len(class_names) == 0:
        raise RuntimeError(f"No class subfolders found in {dataset_root}")

    print(f"[SCAN] Classes found: {class_names}")

    paths: List[Path] = []
    labels: List[str] = []

    for cname in class_names:
        cdir = dataset_root / cname
        n_c = 0
        for p in cdir.rglob("*"):
            if p.is_file() and p.suffix.lower() in IMG_EXTS:
                paths.append(p)
                labels.append(cname)
                n_c += 1
        print(f"[SCAN] Class '{cname}': {n_c} images")

    print(f"[SCAN] Total images: {len(paths)}")
    return paths, labels, class_names

def _materialize_split(
    split_paths: List[Path],
    split_labels: List[str],
    out_root: Path,
    split_name: str
) -> List[Tuple[str, str, str]]:
    rows: List[Tuple[str, str, str]] = []
    for src, cname in zip(split_paths, split_labels):
        dst_dir = out_root / split_name / cname
        dst_dir.mkdir(parents=True, exist_ok=True)
        dst = dst_dir / src.name
        shutil.copy2(src, dst)
        rows.append((split_name, cname, str(src), str(dst)))
    return rows

def _save_manifest(manifest_path: Path, rows: List[Tuple[str, str, str]]):
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    with open(manifest_path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["split", "class", "src", "dst"])
        for row in rows:
            w.writerow(row)
    print(f"[MANIFEST] Saved to {manifest_path}")

def _make_zip_from_dir(src_dir: Path, zip_path: Path):
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    print(f"[ZIP] Creating {zip_path} from {src_dir}")
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk(src_dir):
            rel_root = os.path.relpath(root, src_dir)
            for fn in files:
                p = Path(root) / fn
                if rel_root == ".":
                    arcname = fn
                else:
                    arcname = str(Path(rel_root) / fn)
                z.write(p, arcname=arcname)

def random_split_and_zip(
    zip_path: Path,
    out_dir: Path,
    test_ratio: float = 0.30,
    seed: int = 42,
    remove_dirs_after_zip: bool = False
) -> Dict:
    out_dir.mkdir(parents=True, exist_ok=True)

    # 1) Unzip original dataset
    unzipped_root = _unzip_to_temp(zip_path, out_dir)

    # 2) Detect dataset root within unzipped content
    dataset_root = _detect_dataset_root(unzipped_root)

    # 3) Scan images
    paths, labels, class_names = _scan_images(dataset_root)

    # 4) Random stratified split
    paths_train, paths_test, labels_train, labels_test = train_test_split(
        paths,
        labels,
        test_size=test_ratio,
        random_state=seed,
        stratify=labels, # Asegura que las clases se distribuyan proporcionalmente
    )

    print(f"[SPLIT] train={len(paths_train)} | test={len(paths_test)} | test_ratio={test_ratio:.2f}")

    train_dir = out_dir / "train"
    test_dir = out_dir / "test"

    # Limpiar directorios de salida antes de materializar el split
    if train_dir.exists():
        print(f"[CLEAN] Removing existing directory: {train_dir}")
        shutil.rmtree(train_dir)
    if test_dir.exists():
        print(f"[CLEAN] Removing existing directory: {test_dir}")
        shutil.rmtree(test_dir)

    # 5) Materialize split (copy files)
    rows: List[Tuple[str, str, str]] = []
    rows += _materialize_split(paths_train, labels_train, out_dir, "train")
    rows += _materialize_split(paths_test, labels_test, out_dir, "test")

    # 6) Save manifest
    _save_manifest(out_dir / "manifest_random_split.csv", rows)

    # 7) Create train.zip and test.zip
    train_zip = out_dir / "train.zip"
    test_zip = out_dir / "test.zip"

    if train_dir.exists():
        _make_zip_from_dir(train_dir, train_zip)
    else:
        print("[ZIP] WARNING: train directory not found, skipping train.zip")

    if test_dir.exists():
        _make_zip_from_dir(test_dir, test_zip)
    else:
        print("[ZIP] WARNING: test directory not found, skipping test.zip")

    # 8) Cleanup
    shutil.rmtree(unzipped_root, ignore_errors=True) # Eliminar la carpeta temporal de descompresión
    if remove_dirs_after_zip:
        print("[CLEAN] Removing train/ and test/ directories after zipping...")
        shutil.rmtree(train_dir, ignore_errors=True)
        shutil.rmtree(test_dir, ignore_errors=True)

    print(f"[DONE] Random split written under: {out_dir}")
    print(f"[DONE] Zips: {train_zip} | {test_zip}")
    return {
        "train_dir": train_dir,
        "test_dir": test_dir,
        "classes": class_names,
        "train_zip": train_zip,
        "test_zip": test_zip,
    }

### **Paso 2: Ejecución de la Función `random_split_and_zip`**

Aquí llamas a la función `random_split_and_zip` con tus rutas y parámetros deseados. Observa que el `test_ratio` de `0.30` significa que el 30% de las imágenes irá al conjunto de prueba y el 70% restante al de entrenamiento, lo que equivale a tu solicitud de split 70/30.

In [ ]:
from pathlib import Path

# Define las rutas de entrada y salida
original_zip_path = Path("/content/drive/MyDrive/TESIS/datasetclean.zip")
output_split_dir = Path("/content/drive/MyDrive/TESIS/splits/random_split_clean_test") # Un nuevo directorio para esta ejecución

# Ejecuta la función de split aleatorio
result_split_info = random_split_and_zip(
    zip_path=original_zip_path,
    out_dir=output_split_dir,
    test_ratio=0.30, # 30% para test, 70% para train
    seed=42,
    remove_dirs_after_zip=False # Mantener las carpetas train/ y test/ después de zipear si es útil para depurar
)

print("\nInformación del split generado:")
print(result_split_info)

### **Paso 3: Verificación de los Conteos de Imágenes**

Esta celda extrae los archivos `train.zip` y `test.zip` generados en el paso anterior y cuenta las imágenes que contienen. Esto te permitirá confirmar que los conteos corresponden al split de 70/30 (aproximadamente 2144 para train y 920 para test si el total es 3064).

In [ ]:
import os
import shutil
from pathlib import Path
import zipfile

# Rutas a los ZIPs generados por el paso anterior
train_zip_gdrive = result_split_info['train_zip']
test_zip_gdrive = result_split_info['test_zip']

# Carpeta local para la extracción y conteo
local_verification_dir = Path("/content/temp_verification_unzipped")
local_verification_dir.mkdir(parents=True, exist_ok=True)

unzipped_train_dir = local_verification_dir / "train_extracted"
unzipped_test_dir = local_verification_dir / "test_extracted"

IMG_EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}

def count_images_in_dir(directory: Path) -> int:
    count = 0
    if not directory.is_dir():
        return 0
    for root, _, files in os.walk(directory):
        for file in files:
            if Path(file).suffix.lower() in IMG_EXTS:
                count += 1
    return count

print(f"[VERIFY] Extracting {train_zip_gdrive} to {unzipped_train_dir}...")
unzipped_train_dir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(train_zip_gdrive, 'r') as zip_ref:
    zip_ref.extractall(unzipped_train_dir)

print(f"[VERIFY] Extracting {test_zip_gdrive} to {unzipped_test_dir}...")
unzipped_test_dir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(test_zip_gdrive, 'r') as zip_ref:
    zip_ref.extractall(unzipped_test_dir)

num_train_images_verified = count_images_in_dir(unzipped_train_dir)
num_test_images_verified = count_images_in_dir(unzipped_test_dir)

print(f"\n[VERIFIED RESULT] Imágenes en train.zip (extraídas): {num_train_images_verified}")
print(f"[VERIFIED RESULT] Imágenes en test.zip (extraídas): {num_test_images_verified}")

# Limpiar la carpeta de verificación temporal
shutil.rmtree(local_verification_dir, ignore_errors=True)
print("[CLEANUP] Carpeta de verificación temporal eliminada.")